# Emotion detection from your own voice (best CREMA-D model) + SHAP explanation
1. Reads your results and picks the **highest-accuracy model** (your run: fold 2). 2. Loads its `foldN.pt` weights. 3. Record your voice -> **emotion + confidence** -> **SHAP explanation** (which moments and which frequency ranges drove the decision, plus the Transformer's own attention).

**You need:** `fold2.pt` (from Colab `/content/models/fold2.pt`) in the same folder as this notebook, plus `ser_results.zip` (or the `ser_results` folder). Works on CPU; GPU is faster.
**Be realistic:** the model was trained on acted English sentences of 91 actors. Its measured accuracy on unseen actors is ~75% (best fold ~79%), so for your own voice expect a good, not perfect, result. Speak expressively for 2-4 s in a quiet room, e.g. *"It's eleven o'clock"*, *"That is exactly what happened"*, *"I'm on my way to the meeting"*.

In [ ]:
%pip install -q torch transformers librosa soundfile shap matplotlib pandas ipywidgets sounddevice

In [ ]:
import os, glob, json, math, time, zipfile, warnings, numpy as np, pandas as pd, librosa, soundfile, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from IPython.display import display
# ===================== EDIT ONLY THIS BLOCK =====================
RESULTS_DIR = r"ser_results"      # folder (or ser_results.zip next to this notebook) with finetune_fold_results.csv
MODEL_DIR   = r"."                # folder that contains fold0.pt ... fold4.pt (at least the best fold's file, e.g. fold2.pt)
USE_ALL_FOLD_MODELS = False       # True = average every foldN.pt you have (usually a bit more robust); False = only the best-accuracy model
MODEL_NAME  = "microsoft/wavlm-base-plus"   # backbone you trained with
# =================================================================
DEBUG=os.environ.get("SER_DEBUG","0")=="1"
if DEBUG: RESULTS_DIR,MODEL_DIR=os.environ["SER_RES"],os.environ["SER_MODELS"]
EMOS=["ANG","DIS","FEA","HAP","NEU","SAD"]; FULL={"ANG":"Anger","DIS":"Disgust","FEA":"Fear","HAP":"Happiness","NEU":"Neutral","SAD":"Sadness"}
SR,MAXS=16000,56000
import torch, torch.nn as nn, torch.nn.functional as F
dev="cuda" if torch.cuda.is_available() else "cpu"; use_amp=dev=="cuda"; print("device:",dev)

# ---- 1) which model has the highest accuracy? (reads your results) ----
if not os.path.isdir(RESULTS_DIR) and os.path.exists("ser_results.zip"): zipfile.ZipFile("ser_results.zip").extractall(RESULTS_DIR)
BEST_FOLD=None
p=os.path.join(RESULTS_DIR,"finetune_fold_results.csv")
if os.path.exists(os.path.join(RESULTS_DIR,"final_model_comparison.csv")):
    print("Method comparison (actor-independent accuracy %):"); display(pd.read_csv(os.path.join(RESULTS_DIR,"final_model_comparison.csv")).round(2))
if os.path.exists(p):
    fr=pd.read_csv(p); display(fr.round(4)); BEST_FOLD=int(fr.loc[fr.acc.idxmax(),"fold"])
    print(f"-> best single model: fold {BEST_FOLD} (acc {fr.acc.max():.4f}); 5-fold mean acc {fr.acc.mean():.4f} +- {fr.acc.std():.4f}")
else: BEST_FOLD=2; print("results csv not found, defaulting to fold",BEST_FOLD)
def find_model(f):
    for d in [MODEL_DIR,".","models","/content/models"]:
        q=os.path.join(d,f"fold{f}.pt")
        if os.path.exists(q): return q
cands=[f for f in range(5) if find_model(f)] if USE_ALL_FOLD_MODELS else [BEST_FOLD]
MODEL_FILES=[find_model(f) for f in cands if find_model(f)]
assert MODEL_FILES, f"fold{BEST_FOLD}.pt not found. Copy it from Colab (/content/models/fold{BEST_FOLD}.pt) into the folder: {os.path.abspath(MODEL_DIR)}"
print("model file(s):",MODEL_FILES)

In [ ]:
from transformers import WavLMModel, WavLMConfig
def build_backbone():
    if DEBUG:
        return WavLMModel(WavLMConfig(hidden_size=32,num_hidden_layers=2,num_attention_heads=2,intermediate_size=64,conv_dim=(16,)*7,num_feat_extract_layers=7,num_buckets=32,max_bucket_distance=64,num_conv_pos_embeddings=8,num_conv_pos_embedding_groups=2,layerdrop=0.0))
    return WavLMModel(WavLMConfig.from_pretrained(MODEL_NAME,layerdrop=0.0,apply_spec_augment=False))      # architecture only; weights come from your .pt file
class SER(nn.Module):       # identical to the training notebook
    def __init__(self,n_cls=6,hid=256,p=0.2):
        super().__init__(); self.enc=build_backbone(); self.enc.freeze_feature_encoder()
        L=self.enc.config.num_hidden_layers+1; D=self.enc.config.hidden_size
        self.lw=nn.Parameter(torch.zeros(L)); self.att=nn.Sequential(nn.Linear(D,128),nn.Tanh(),nn.Linear(128,1))
        self.head=nn.Sequential(nn.LayerNorm(D),nn.Dropout(p),nn.Linear(D,hid),nn.GELU(),nn.Dropout(p),nn.Linear(hid,n_cls))
    def forward(self,x,l):
        H=torch.stack(self.enc(x,output_hidden_states=True).hidden_states,0); w=torch.softmax(self.lw,0); h=(w[:,None,None,None]*H).sum(0); T=h.shape[1]
        fl=torch.clamp((l.to(x.device)-400)//320+1,1,T); m=torch.arange(T,device=x.device)[None]<fl[:,None]
        a=torch.softmax(self.att(h).squeeze(-1).float().masked_fill(~m,-1e4),1)
        return self.head((a.unsqueeze(-1)*h.float()).sum(1)),a
def load_ckpt(model,path):
    sd=torch.load(path,map_location="cpu"); sd={k:(v.float() if v.is_floating_point() else v) for k,v in sd.items()}; msd=model.state_dict(); new={}
    for k,v in sd.items():           # tolerate weight-norm key naming differences between transformers/torch versions
        a1=k.replace("weight_g","parametrizations.weight.original0").replace("weight_v","parametrizations.weight.original1")
        a2=k.replace("parametrizations.weight.original0","weight_g").replace("parametrizations.weight.original1","weight_v")
        new[k if k in msd else a1 if a1 in msd else a2 if a2 in msd else k]=v
    r=model.load_state_dict(new,strict=False); miss=[k for k in r.missing_keys if "masked_spec_embed" not in k]
    assert not miss, f"checkpoint mismatch, missing: {miss[:5]}"; return model
models=[]
for pth in MODEL_FILES:
    m=SER().to(dev); load_ckpt(m,pth); models.append(m.eval()); print("loaded",pth)

In [ ]:
def prep(x_np,l_np):
    l=torch.as_tensor(l_np).long(); x=torch.as_tensor(x_np[:,:int(l.max())],dtype=torch.float32); m=(torch.arange(x.shape[1])[None]<l[:,None]).float()
    mean=(x*m).sum(1,keepdim=True)/l[:,None]; var=(((x-mean)*m)**2).sum(1,keepdim=True)/l[:,None]; return ((x-mean)/(var.sqrt()+1e-5))*m,l
def clean_audio(audio,sr):
    """same preprocessing as training: 16 kHz mono, trim silence (top_db=30), max 3.5 s (loudest 3.5 s window if longer)"""
    audio=np.asarray(audio,dtype=np.float32); audio=audio.mean(1) if audio.ndim>1 else audio
    if sr!=SR: audio=librosa.resample(audio,orig_sr=sr,target_sr=SR)
    a,_=librosa.effects.trim(audio,top_db=30)
    if len(a)<SR//2: raise ValueError("recording is too short or silent - speak for 2-4 seconds, closer to the mic")
    if len(a)>MAXS:
        e=np.convolve(a**2,np.ones(SR//4),"valid"); st=int(np.argmax(np.convolve(e,np.ones(MAXS-SR//4+1),"valid"))); a=a[st:st+MAXS]
    return a
@torch.no_grad()
def probs_of(batch,bs=48,return_att=False):
    """batch: list/array of equal-length float32 waveforms -> (k,6) probabilities averaged over loaded model(s)"""
    X=np.asarray(batch,dtype=np.float32); out=[]; atts=[]
    for s in range(0,len(X),bs):
        xb,lb=prep(X[s:s+bs],np.full(len(X[s:s+bs]),X.shape[1])); ps=[]
        for m in models:
            with torch.autocast("cuda",dtype=torch.float16,enabled=use_amp): lo,a=m(xb.to(dev),lb)
            ps.append(torch.softmax(lo.float(),1).cpu().numpy()); atts.append(a.float().cpu().numpy())
        out.append(np.mean(ps,0))
    return (np.concatenate(out),np.mean(atts,0)) if return_att else np.concatenate(out)
def predict(audio,sr=SR):
    a=clean_audio(audio,sr); p,att=probs_of(a[None],return_att=True); return p[0],a,att[0]

## SHAP explanation engine

In [ ]:
import shap
NSEG,NSAMPLES=10,300 if not DEBUG else 20
EDGES=[0,300,600,1000,1500,2200,3200,4500,8001]; NB=len(EDGES)-1
def _sv(ex,M,ns):
    sv=ex.shap_values(np.ones((1,M)),nsamples=ns,silent=True); sv=np.stack(sv,-1) if isinstance(sv,list) else np.asarray(sv); return sv.reshape(M,-1)    # (M,6)
def shap_time(a,ns=NSAMPLES):
    n=len(a); b=np.linspace(0,n,NSEG+1).astype(int)
    def f(Z):
        X=np.tile(a,(len(Z),1))
        for j in range(NSEG): X[np.where(np.asarray(Z)[:,j]==0)[0],b[j]:b[j+1]]=0           # removed segment = silence
        return probs_of(X)
    ex=shap.KernelExplainer(f,np.zeros((1,NSEG))); return _sv(ex,NSEG,ns),b
def shap_bands(a,ns=NSAMPLES):
    S=librosa.stft(a,n_fft=512,hop_length=128); fr=librosa.fft_frequencies(sr=SR,n_fft=512)
    def f(Z):
        X=[]
        for z in np.asarray(Z):
            M=np.ones(S.shape,np.float32)
            for j in range(NB):
                if z[j]==0: M[(fr>=EDGES[j])&(fr<EDGES[j+1])]=0                          # removed band = filtered out
            X.append(librosa.istft(S*M,hop_length=128,length=len(a)))
        return probs_of(np.array(X))
    ex=shap.KernelExplainer(f,np.zeros((1,NB))); return _sv(ex,NB,ns)
def explain(audio,sr=SR,with_shap=True):
    p,a,att=predict(audio,sr); order=np.argsort(-p); k,k2=int(order[0]),int(order[1]); n=len(a); t=np.arange(n)/SR
    print("="*60); print(f"DETECTED EMOTION : {FULL[EMOS[k]].upper()}"); print(f"CONFIDENCE       : {p[k]*100:.1f}%   (runner-up: {FULL[EMOS[k2]]} {p[k2]*100:.1f}%)")
    if p[k]<0.5 or p[k]-p[k2]<0.15: print("NOTE: model is not very sure - the emotion is ambiguous or not strongly expressed; try again with a clearer, more expressive sentence.")
    print("="*60)
    fig=plt.figure(figsize=(14,9)); gs=fig.add_gridspec(3,2,height_ratios=[1,1.1,1.1])
    ax=fig.add_subplot(gs[0,0]); ax.barh([FULL[e] for e in EMOS][::-1],(p*100)[::-1],color=["#d62728" if i==k else "#8c8c8c" for i in range(6)][::-1]); ax.set_xlim(0,100); ax.set_xlabel("%"); ax.set_title("Predicted probabilities")
    if not with_shap: plt.tight_layout(); plt.show(); return p
    sv_t,b=shap_time(a); sv_b=shap_bands(a); sk=sv_t[:,k]; ax2=fig.add_subplot(gs[0,1]); ax2.plot(t,a,lw=.4,color="#555")
    mx=np.abs(sk).max()+1e-9
    for j in range(NSEG): ax2.axvspan(b[j]/SR,b[j+1]/SR,color=(plt.cm.RdBu_r((sk[j]/mx+1)/2)),alpha=.45)
    ax2.set_title(f"Your voice: red = pushes TOWARD {FULL[EMOS[k]]}, blue = pushes AWAY"); ax2.set_xlabel("time (s)")
    ax3=fig.add_subplot(gs[1,0]); w=.38; xs=np.arange(NSEG); ax3.bar(xs-w/2,sv_t[:,k],w,label=FULL[EMOS[k]],color="#d62728"); ax3.bar(xs+w/2,sv_t[:,k2],w,label=FULL[EMOS[k2]],color="#1f77b4")
    ax3.set_xticks(xs); ax3.set_xticklabels([f"{b[j]/SR:.1f}s" for j in range(NSEG)],fontsize=7); ax3.axhline(0,color="k",lw=.5); ax3.set_title("SHAP by TIME segment (probability contribution)"); ax3.legend(fontsize=8)
    ax4=fig.add_subplot(gs[1,1]); lab=[f"{EDGES[j]/1000:.1f}-{min(EDGES[j+1],8000)/1000:.1f}k" for j in range(NB)]; xb=np.arange(NB)
    ax4.bar(xb-w/2,sv_b[:,k],w,label=FULL[EMOS[k]],color="#d62728"); ax4.bar(xb+w/2,sv_b[:,k2],w,label=FULL[EMOS[k2]],color="#1f77b4"); ax4.set_xticks(xb); ax4.set_xticklabels(lab,fontsize=7); ax4.axhline(0,color="k",lw=.5)
    ax4.set_title("SHAP by FREQUENCY band (Hz) - which sound range mattered"); ax4.legend(fontsize=8)
    ax5=fig.add_subplot(gs[2,0]); at=np.array([s.mean() if len(s) else 0 for s in np.array_split(att[:max(1,int(np.clip((n-400)//320+1,1,len(att))))],NSEG)]); ax5.plot(xs,at/at.max(),marker="s",label="model attention"); ax5.plot(xs,sk/(np.abs(sk).max()+1e-9),marker="o",label="SHAP (predicted class)")
    ax5.set_xticks(xs); ax5.set_xticklabels([f"{b[j]/SR:.1f}s" for j in range(NSEG)],fontsize=7); ax5.set_title("SHAP vs the Transformer's own attention over time"); ax5.legend(fontsize=8); ax5.axhline(0,color="k",lw=.5)
    ax6=fig.add_subplot(gs[2,1]); ax6.axis("off")
    top_t=np.argsort(-sk)[:2]; neg_t=np.argsort(sk)[:1]; top_b=np.argsort(-sv_b[:,k])[:2]; neg_b=np.argsort(sv_b[:,k])[:1]
    txt=(f"WHY {FULL[EMOS[k]].upper()}?\n\n- Strongest evidence at {', '.join(f'{b[j]/SR:.1f}-{b[j+1]/SR:.1f}s' for j in top_t)}\n  (pushing probability up by {', '.join(f'{sk[j]*100:+.0f}%' for j in top_t)}).\n"
         f"- Most useful sound range: {', '.join(lab[j]+'Hz' for j in top_b)} ({', '.join(f'{sv_b[j,k]*100:+.0f}%' for j in top_b)}).\n"
         f"- Worked against it: {b[neg_t[0]]/SR:.1f}-{b[neg_t[0]+1]/SR:.1f}s ({sk[neg_t[0]]*100:+.0f}%), {lab[neg_b[0]]}Hz ({sv_b[neg_b[0],k]*100:+.0f}%).\n"
         f"- Compared with {FULL[EMOS[k2]]}: time-evidence {sv_t[:,k].sum()*100:+.0f}% vs {sv_t[:,k2].sum()*100:+.0f}%.\n\n(SHAP value = change in the model's probability when that\npart of your voice is removed; values add up across parts.)")
    ax6.text(0,1,txt,va="top",fontsize=10,family="monospace"); plt.tight_layout(); plt.savefig("last_explanation.png",dpi=130); plt.show(); print("saved last_explanation.png"); print(txt.replace("\n","\n")); return p
def predict_file(path,with_shap=True): a,sr=soundfile.read(path,dtype="float32"); return explain(a,sr,with_shap)

## Record your voice -> emotion + confidence + why

In [ ]:
# ---------------- record your voice ----------------
try:
    import ipywidgets as W, sounddevice as sd
    frames,state=[],{"stream":None}; out=W.Output(); chk=W.Checkbox(value=True,description="explain with SHAP (takes ~10-60 s)")
    b_start=W.Button(description="Start recording",button_style="success",icon="microphone"); b_stop=W.Button(description="Stop & detect emotion",button_style="danger",icon="stop",disabled=True)
    def _cb(indata,fr,t,st): frames.append(indata.copy())
    def _start(_):
        frames.clear(); out.clear_output(); state["stream"]=sd.InputStream(samplerate=SR,channels=1,dtype="float32",callback=_cb); state["stream"].start(); b_start.disabled=True; b_stop.disabled=False
        with out: print("recording ... say a sentence with clear emotion (2-4 s), then press Stop")
    def _stop(_):
        state["stream"].stop(); state["stream"].close(); b_start.disabled=False; b_stop.disabled=True; audio=np.concatenate(frames)[:,0]; soundfile.write("my_voice.wav",audio,SR)
        with out:
            out.clear_output()
            try: explain(audio,SR,chk.value)
            except Exception as ex: print("error:",ex)
    b_start.on_click(_start); b_stop.on_click(_stop); display(W.HBox([b_start,b_stop,chk]),out)
except Exception as ex:
    print("Microphone widget unavailable:",ex)
print("Alternative (works anywhere): record a .wav with any app, then run   predict_file('my_voice.wav')")
def record_fixed(seconds=4,with_shap=True):
    import sounddevice as sd; print("recording",seconds,"s ... speak now"); a=sd.rec(int(seconds*SR),samplerate=SR,channels=1,dtype="float32"); sd.wait(); soundfile.write("my_voice.wav",a[:,0],SR); return explain(a[:,0],SR,with_shap)